# Survey Stimulus Generation — Attestation Trust Study

**AI assistance disclosure:** This stimulus-generation tooling was built with the assistance of an AI assistant (Claude) for the code scaffolding (parsing, validation, file output) and tutoring how to use the API and prompt correctly. The generation prompt, the attestation-display wording, and all curation decisions are the author's own (Harry Staley). Per the study's documented method, an LLM generates candidate stimuli which the author then curates. Use of generative AI follows the CS 6795 course policy.

In [68]:
from __future__ import annotations

import json
import re
import time
from collections import Counter
from typing import List, Tuple, TypedDict
from pathlib import Path

import pandas as pd
from openai import OpenAI
from IPython.display import display
from dotenv import load_dotenv
from datetime import datetime
load_dotenv()
if not load_dotenv():
    print("WARNING: .env not found; run setup_env.py to create it.")
print(f"Key loaded: {load_dotenv()}")
print("NOTE: Be sure that you have a .env file with your OpenAI API key.")

Key loaded: True
NOTE: Be sure that you have a .env file with your OpenAI API key.


In [69]:
MODEL: str = "gpt-5.5"            # model name
# NOTE: Temperature is not available in gpt-5.5, but it is in others.
# TEMPERATURE: float = 0.7        # controls randomness; higher = more varied output
MAX_RETRIES: int = 5            # how many times to retry on hard failure
LENGTH_RATIO_WARN: float = 0.25 # warn if answers differ >25% in length
SENTENCE_DIFF_WARN: int = 1     # warn if sentence counts differ by >1

In [70]:
class Stem(TypedDict):
    """Schema for one generated survey stimulus."""
    stem_id: int
    stakes: str
    category: str
    consequence_type: str
    fact_structure: str
    topic: str
    question_text: str
    correct_answer: str
    incorrect_answer: str
    source_name: str
    source_citation: str
    source_url: str
    ground_truth_note: str

# Verifies that the JSON object has the required keys as defined in the Stem class.
REQUIRED_KEYS: set[str] = {
    "stem_id", "stakes", "category", "consequence_type", "fact_structure",
    "topic", "question_text",
    "correct_answer", "incorrect_answer", "source_name",
    "source_citation", "source_url", "ground_truth_note",
}

In [71]:
prompt_template = Path("generation_prompt.md").read_text(encoding="utf-8")
GENERATION_PROMPT = prompt_template.format(
    schema_fields=", ".join(Stem.__annotations__)
)

In [72]:
def attestation_text(att_level: str, item: Stem) -> str:
    """Return the rendered attestation display for a given attestation level."""
    if att_level == "none":
        return ""
    if att_level == "weak":
        return f"Source: {item['source_name']} — {item['source_citation']}"
    return (f"Source: {item['source_name']} — {item['source_citation']}\n"
            f"Publisher verified ({item['source_name']})\n"
            f"Document unaltered since publication\n"
            f"Independently checked for relevance")

In [73]:
def parse_json(raw_text: str) -> List[Stem]:
    """Parse model output into stems; recover the [...] array if wrapped."""
    raw_text = raw_text.strip()
    raw_text = re.sub(r"^```(?:json)?|```$", "", raw_text, flags=re.MULTILINE).strip()
    try:
        return json.loads(raw_text)
    except json.JSONDecodeError:
        start, end = raw_text.find("["), raw_text.rfind("]") + 1
        if start == -1 or end == 0:
            raise
        return json.loads(raw_text[start:end])

In [74]:
def sentence_count(text: str) -> int:
    """Rough sentence count, robust to decimals/abbreviations (for warnings only)."""
    if not text.strip():
        return 0
    t = re.sub(r"\d+\.\d+", "0", text)
    for abbr in ("Dr.", "Mr.", "Mrs.", "Ms.", "U.S.", "U.K.", "e.g.", "i.e.",
                 "etc.", "mg.", "mL.", "vs.", "Inc.", "Ltd.", "Fig.", "No."):
        t = t.replace(abbr, abbr.replace(".", ""))
    return max(len(re.findall(r"[.!?]+", t)), 1)

In [75]:
def validate_stems(items: List[Stem]) -> Tuple[List[str], List[str]]:
    """Return (errors, warnings). Errors trigger retry; warnings flag for curation."""
    errors: List[str] = []
    warnings: List[str] = []

    if len(items) != 12:
        errors.append(f"Expected 12 stems, found {len(items)}.")
    low = sum(x.get("stakes") == "low" for x in items)
    high = sum(x.get("stakes") == "high" for x in items)
    if low != 6 or high != 6:
        errors.append(f"Expected 6 low / 6 high; found {low} low / {high} high.")
    if sorted(x.get("stem_id", -1) for x in items) != list(range(1, 13)):
        errors.append("stem_id values must be 1..12 with no gaps/dupes.")

    topics: List[str] = []
    for item in items:
        sid = item.get("stem_id", "?")
        if set(item.keys()) != REQUIRED_KEYS:
            errors.append(f"Stem {sid} schema mismatch.")
            continue
        topics.append(item["topic"].lower())
        ca, ia = item["correct_answer"], item["incorrect_answer"]
        if ia.strip() == "REFUSED_NEEDS_MANUAL" or not ia.strip():
            errors.append(f"Stem {sid} incorrect_answer REFUSED -- build manually.")
            continue
        if abs(sentence_count(ca) - sentence_count(ia)) > SENTENCE_DIFF_WARN:
            warnings.append(f"Stem {sid}: sentence-count mismatch (review).")
        if max(len(ca), len(ia)) and abs(len(ca) - len(ia)) / max(len(ca), len(ia)) > LENGTH_RATIO_WARN:
            warnings.append(f"Stem {sid}: answer-length mismatch (review).")

    dups = [t for t, c in Counter(topics).items() if c > 1]
    if dups:
        errors.append(f"Duplicate topics: {dups}")
    return errors, warnings

In [76]:
# Test the whole logic chain with fake data -- no API, no cost.
_mock = [
    {"stem_id": i, "stakes": "low" if i <= 6 else "high",
     "category": "Consumer/retail facts" if i <= 6 else "Financial penalty/loss",
     "consequence_type": "financial",
     "fact_structure": "threshold-amount",
     "topic": f"topic{i}",
     "question_text": "Q?", "correct_answer": "A true statement here.",
     "incorrect_answer": "A false statement here.", "source_name": "Src",
     "source_citation": "Doc, src.org", "source_url": "https://src.org",
     "ground_truth_note": "note"}
    for i in range(1, 13)
]
_errors, _warnings = validate_stems(_mock)
print("errors:", _errors)
print("warnings:", _warnings)
assert not _errors, "mock should pass structural validation"
print("MOCK PASSED — logic chain works.")

errors: []
warnings: []
MOCK PASSED — logic chain works.


In [77]:
def generate_response() -> List[Stem]:
    """One generation call; stem_ids are assigned in code, not trusted from the model."""
    response = client.responses.create(
        model=MODEL,
        input=GENERATION_PROMPT,
    )
    items = parse_json(response.output_text)
    # assign stem_ids by position -- the model is unreliable at sequential numbering
    for i, item in enumerate(items, start=1):
        item["stem_id"] = i
    return items

In [78]:
client = OpenAI()
_test = client.responses.create(
    model=MODEL,
    input='Return exactly this JSON and nothing else: [{"ok": 1}]',
)
print(repr(_test.output_text))

'[{"ok": 1}]'


In [79]:
def generate_with_retries() -> Tuple[List[Stem], List[str]]:
    """Generate stems; retry on HARD failures, surface warnings for curation."""
    last_errors: List[str] = []
    for attempt in range(1, MAX_RETRIES + 1):
        print("=" * 80)
        print(f"ATTEMPT {attempt}/{MAX_RETRIES}")
        try:
            items = generate_response()
            errors, warnings = validate_stems(items)
            if not errors:
                print("Structural validation passed.")
                if warnings:
                    print(f"\n{len(warnings)} item(s) flagged for human curation:")
                    for w in warnings:
                        print("  ~", w)
                else:
                    print("No curation warnings.")
                print()
                return items, warnings
            print("Hard failures (regenerating):")
            for e in errors:
                print("  -", e)
            last_errors = errors
        except Exception as e:
            print("Generation failed:", str(e))
            last_errors = [str(e)]
        time.sleep(1)
    raise RuntimeError(
        "Generation failed after retries. Last hard failures:\n"
        + "\n".join(last_errors)
        + "\n\nIf failures are REFUSED incorrect answers on sensitive items, "
          "generate the rest and CONSTRUCT those items manually (document it)."
    )


In [80]:
def build_loop_table(items: List[Stem]) -> pd.DataFrame:
    """Expand validated stems into the 72-row Qualtrics loop table."""
    rows: List[dict] = []
    for item in items:
        for att_level in ("none", "weak", "strong"):
            for correctness in ("correct", "incorrect"):
                answer = (item["correct_answer"] if correctness == "correct"
                          else item["incorrect_answer"])
                rows.append({
                    "stem_id": item["stem_id"],
                    "stakes": item["stakes"],
                    "question_text": item["question_text"],
                    "answer_text": answer,
                    "att_level": att_level,
                    "correctness": correctness,
                    "attestation_text": attestation_text(att_level, item),
                })
    return pd.DataFrame(rows)

In [67]:
items, warnings = generate_with_retries()

stems_df = pd.DataFrame(items)
loop_df = build_loop_table(items)

# show results
print("\n" + "=" * 80 + "\nGENERATED STEMS\n" + "=" * 80)
print(stems_df.to_string(index=False))
print("\n" + "=" * 80 + "\nQUALTRICS LOOP TABLE\n" + "=" * 80)
print(loop_df.to_string(index=False))

display(stems_df)
display(loop_df)

# write outputs to a dedicated directory
out = Path("output")
out.mkdir(exist_ok=True)
ts = datetime.now().strftime("%Y-%m-%dT%H%M%S")

stems_df.to_csv(out / f"generated_stems_{ts}.csv", index=False)
loop_df.to_csv(out / f"qualtrics_loop_table_{ts}.csv", index=False)
with open(out / f"generated_stems_{ts}.json", "w", encoding="utf-8") as f:
    json.dump(items, f, indent=2)

metadata = {
    "model": MODEL,
    "timestamp": datetime.now().isoformat(timespec="seconds"),
    "max_retries": MAX_RETRIES,
    "curation_warnings": warnings,
    "note": "Soft warnings indicate items flagged for human curation "
            "(matched length/sentence count), per the documented method.",
}
with open(out / f"generation_metadata_{ts}.json", "w") as f:
    json.dump(metadata, f, indent=2)

print(f"\nSaved to {out}/: generated_stems_{ts}.json, generated_stems_{ts}.csv, "
      f"qualtrics_loop_table_{ts}.csv, generation_metadata_{ts}.json")
if warnings:
    print(f"\n{len(warnings)} item(s) need curation review (see {out}/generation_metadata_{ts}.json).")

ATTEMPT 1/5
Hard failures (regenerating):
  - Expected 6 low / 6 high; found 0 low / 0 high.
ATTEMPT 2/5
Hard failures (regenerating):
  - Expected 6 low / 6 high; found 0 low / 0 high.
ATTEMPT 3/5
Hard failures (regenerating):
  - Expected 6 low / 6 high; found 0 low / 0 high.
ATTEMPT 4/5
Structural validation passed.
No curation warnings.


GENERATED STEMS
 stem_id stakes                                category                                                      consequence_type      fact_structure                                     topic                                                                                                                                                                         question_text                                                                                  correct_answer                                                                            incorrect_answer                                    source_name                                   

,stem_id,stakes,category,consequence_type,fact_structure,topic,question_text,correct_answer,incorrect_answer,source_name,source_citation,source_url,ground_truth_note
0,1,high,Financial penalty/loss,Federal excise tax on excess savings-account c...,threshold/amount,Health savings accounts,"At the federal level, if a health savings acco...",A 6% excise tax applies to the excess amount f...,A 10% excise tax applies to the excess amount ...,Internal Revenue Service,"IRS Publication 969, Health Savings Accounts a...",https://www.irs.gov/publications/p969,IRS Publication 969 states that excess HSA con...
1,2,high,Benefit/coverage forfeiture,Late enrollment penalty and delayed health cov...,eligibility window,Medicare Part B enrollment,"In general, for someone first eligible for Med...",The period lasts seven months: three months be...,The period lasts six months: three months befo...,Medicare.gov,"Medicare.gov, Signing up for Medicare, initial...",https://www.medicare.gov/basics/get-started-wi...,Medicare.gov describes the initial enrollment ...
2,3,high,Legal-right or claim forfeiture,Loss of federal tax-court petition right,deadline,IRS notice of deficiency,"At the federal level, after the IRS mails a no...",The taxpayer generally has 90 days from the da...,The taxpayer generally has 120 days from the d...,Internal Revenue Service,"IRS, Understanding Your CP3219A Notice, Notice...",https://www.irs.gov/individuals/understanding-...,The IRS states that a taxpayer generally has 9...
3,4,high,Cybersecurity,Potentially unlimited liability for unauthoriz...,rights/entitlements,Lost or stolen debit cards,"Under federal rules, if a debit-card holder wa...",The holder can be liable for all unauthorized ...,The holder can be liable for no more than $500...,Federal Trade Commission,"FTC Consumer Advice, Lost or Stolen Credit, AT...",https://consumer.ftc.gov/articles/lost-or-stol...,The FTC explains that reporting a lost or stol...
4,5,high,Health conditions or medical conditions,Medication overdose risk,threshold/amount,Acetaminophen use,For a generally healthy adult using over-the-c...,The total from all medicines should not exceed...,The total from all medicines should not exceed...,U.S. Food and Drug Administration,"FDA, Acetaminophen Information, consumer dosin...",https://www.fda.gov/drugs/information-drug-cla...,FDA consumer information states that adults sh...
5,6,high,Food safety,Foodborne illness risk,threshold/amount,Safe cooking temperature for fish,"For food safety, to what minimum internal temp...",Fin fish should generally be cooked to an inte...,Fin fish should generally be cooked to an inte...,FoodSafety.gov,"FoodSafety.gov, Safe Minimum Internal Temperat...",https://www.foodsafety.gov/food-safety-charts/...,The federal FoodSafety.gov safe-temperature ch...
6,7,low,Shipping/postal facts,Minor postage shortfall or delivery delay,threshold/amount,First-Class Mail letters,"For a domestic First-Class Mail letter, up to ...",One standard Forever stamp generally covers a ...,One standard Forever stamp generally covers a ...,United States Postal Service,"USPS, First-Class Mail service information",https://www.usps.com/ship/first-class-mail.htm,USPS states that First-Class Mail letters usin...
7,8,low,Everyday financial-convenience facts,Minor delay in access to deposited funds,threshold/amount,Check deposit holds,"When a bank places a hold on a check deposit, ...",The bank must generally make the first $225 av...,The bank must generally make the first $200 av...,Consumer Financial Protection Bureau,"CFPB, Ask CFPB: How quickly can I get money af...",https://www.consumerfinance.gov/ask-cfpb/how-q...,"The CFPB explains that, when a check hold appl..."
8,9,low,Travel-logistics facts,Minor rebooking hassle or baggage delay,deadline,Checked baggage for intercity rail travel,For a U.S. intercity passenger train trip wher...,Checked baggage generally must be checked at l...,Checked baggage generally must be checked at l...,Amtrak,"Amtrak, 

,stem_id,stakes,question_text,answer_text,att_level,correctness,attestation_text
0,1,high,"At the federal level, if a health savings acco...",A 6% excise tax applies to the excess amount f...,none,correct,
1,1,high,"At the federal level, if a health savings acco...",A 10% excise tax applies to the excess amount ...,none,incorrect,
2,1,high,"At the federal level, if a health savings acco...",A 6% excise tax applies to the excess amount f...,weak,correct,Source: Internal Revenue Service — IRS Publica...
3,1,high,"At the federal level, if a health savings acco...",A 10% excise tax applies to the excess amount ...,weak,incorrect,Source: Internal Revenue Service — IRS Publica...
4,1,high,"At the federal level, if a health savings acco...",A 6% excise tax applies to the excess amount f...,strong,correct,Source: Internal Revenue Service — IRS Publica...
...,...,...,...,...,...,...,...
67,12,low,How many standard time zones are used by the 5...,The 50 U.S. states use five standard time zone...,none,incorrect,
68,12,low,How many standard time zones are used by the 5...,The 50 U.S. states use six standard time zones...,weak,correct,Source: National Institute of Standards and Te...
69,12,low,How many standard time zones are used by the 5...,The 50 U.S. states use five standard time zone...,weak,incorrect,Source: National Institute of Standards and Te...
70,12,low,How many standard time zones are used by the 5...,The 50 U.S. states use six standard time zones...,strong,correct,Source: National Institute of Standards and Te...



Saved to output/: generated_stems_2026-06-21T191356.json, generated_stems_2026-06-21T191356.csv, qualtrics_loop_table_2026-06-21T191356.csv, generation_metadata_2026-06-21T191356.json
